Gotta make some tuning curves for homings. 
Two types of plots:
- tuning curves, by condition sorted for that condition and the sorting applied to the other two conditions
- scatter of peak firing of each neuron across conditions

In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

#JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# 
experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

#
trials = [[1,3],[1,5,7],[1],[2],
    [2,3],[1,3],
    [1,3,4],[1,2,3],[1,2],[1,3],
    [4,5],[1,3],[3,4],[4,5],[2,5],
    [1,3],[1,2],[1,3],[5,7,8],[3,5],
]

In [118]:
%load_ext autoreload
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.escape_utils import load, check_not_list, load_homing, compute_dist_shelt, compute_escape_trajectory, compress_vars, discretize_x_axis, firing_by_bin

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
from scipy.stats import zscore
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [366]:
%autoreload
compression_var = ['y_pos', 'distance_shelter', 'escape','speed']
for i, exp in enumerate(experiments_objects):
    session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip = load(exp)
    ons, offs = load_homing(session)
    for comp in compression_var:
        nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
        esc_var, escape_matrix, cond, h_start = extract_homing_time(session, ons, offs, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, comp)
        peak_firing_condition, tuning, xval = peak_firing_by_condition(esc_var, escape_matrix, cond, h_start)
        plot_pref_firing_condition(peak_firing_condition, xval, comp, nickname + '_peak_firing')
        tuning_curve_by_condition(tuning, xval, comp, esc_var, cond, nickname + '_tuning')

C:\Users\Jasmine\AppData\Local\Temp\ipykernel_7764\3297587258.py:14: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  axs[j,i].imshow(plot_t[isort,:], cmap="gray_r", vmin = 0, vmax = 1.2, aspect="auto", interpolation = "none")
C:\Users\Jasmine\AppData\Local\Temp\ipykernel_7764\3297587258.py:14: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  axs[j,i].imshow(plot_t[isort,:], cmap="gray_r", vmin = 0, vmax = 1.2, aspect="auto", interpolation = "none")
C:\Users\Jasmine\AppData\Local\Temp\ipykernel_7764\3297587258.py:14: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  axs[j,i].imshow(plot_t[isort,:], cmap="gray_r", vmin = 0, vmax = 1.2, aspect="auto", interpolation = "none")
C:\Users\Jasmine\AppData\Local\Temp\ipykernel_7764\3297587258.py:14: UserWarning: Attempting to set identical lo

ValueError: Input contains NaN.

In [364]:
def tuning_curve_by_condition(tuning, xval, comp, esc_var, cond, nickname):
    """A plot of tuning curves, by condition sorted for that condition and the sorting applied to the other two conditions"""
    condy = ['shelter only', 'barrier','flipped barrier']
    fig, axs = plt.subplots(3,3,figsize = (9,9))
    fig.suptitle(comp)

    for j, cc in enumerate(condy):
        t = tuning[j][xval[:,j] == 1,:]
        idx = np.argmax(t, axis = 1)
        isort = np.argsort(idx)
        axs[j,0].set_ylabel('neurons sorted by ' + cc)
        for i, c in enumerate(condy):
            plot_t = tuning[i][xval[:,j] == 1,:]
            axs[j,i].imshow(plot_t[isort,:], cmap="gray_r", vmin = 0, vmax = 1.2, aspect="auto", interpolation = "none")
            axs[j,i].set_title(c)

    plt.tight_layout()
    dump_path = "Z:/Jasmine_Laurence/homing/peak_firing_condition"
    fig.savefig(dump_path + "/" + nickname + ".png")
    plt.close()

In [363]:
def plot_pref_firing_condition(peak_firing_condition, xval, comp, nickname):
    c = ['shelter only', 'barrier','flipped barrier']
    fig, axs = plt.subplots(3,3,figsize = (9,9))
    fig.suptitle(comp)

    axlim = [0,np.amax(peak_firing_condition)]

    for j, cc in enumerate(c):
        pfc = peak_firing_condition[xval[:,j] == 1,:]
        for i, (x, y) in enumerate(zip([0,0,1],[1,2,2])):
            axs[j,i].scatter(pfc[:,x],pfc[:,y], s = 3)
            axs[j,i].plot(axlim,axlim,'--k')
            axs[j,i].set_xlabel(c[x])
            axs[j,i].set_ylabel(c[y])
            axs[j,i].set_xlim(axlim)
            axs[j,i].set_ylim(axlim)

    plt.tight_layout()
    dump_path = "Z:/Jasmine_Laurence/homing/peak_firing_condition"
    fig.savefig(dump_path + "/" + nickname + ".png")
    plt.close()

In [356]:
def peak_firing_by_condition(esc_var, escape_matrix, cond, h_start):
    """What is the tuning curve and peak firing bin per condition?"""
    peak_firing_condition = np.zeros((np.shape(escape_matrix)[0], len(np.unique(cond))))
    tuning_by_cond = []
    xval = np.zeros((np.shape(escape_matrix)[0], len(np.unique(cond))))
    for i in np.unique(cond):
        # start by condition
        start = [x for x in h_start if cond[x] == i]
        tuning_matrix, xval[:,int(i)] = create_xval_tuning_curve(esc_var[cond == i], 
                                                                 escape_matrix[:,cond == i], 
                                                                 [x-start[0] for x in start], # resetting the timestamp of homing starts to align to the start of the condition
                                                                 bins = int(np.amax(esc_var)+1))
        # bins = esc_var[cond == i]
        peak_firing = np.argmax(tuning_matrix, axis = 1)
        peak_firing_condition[:,int(i)] = peak_firing#bins[peak_firing]
        tuning_by_cond.append(tuning_matrix)
    return peak_firing_condition, tuning_by_cond, xval.astype(int)

In [337]:
def extract_homing_time(session, ons, offs, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, compression_var, interpolation = True, no_stationary = False):
    """Tuning for each neuron is measured by compression_var"""
    # extract the time around escapes
    start = [0]
    h_start = [0]
    
    esc_ons = check_not_list(session.audio.onset_frames)
    st = [x*40 for x in check_not_list(session.audio.stimulus_durations)]
    esc_offs = (np.add(esc_ons, st)).astype(int)

    ons = np.sort(np.append(check_not_list(ons), esc_ons))
    offs = np.sort(np.append(check_not_list(offs), esc_offs))

    mult = 1
    if interpolation: mult = 2
    escape_matrix = np.zeros((np.shape(frame_by_cluster_matrix)[1],np.sum(offs - ons)*mult)) # x 2 because of interpolation over time
    esc_var = np.zeros(np.sum(offs - ons)*mult)
    in_shelter = np.zeros(np.sum(offs - ons)*mult)
    cond = np.zeros(np.sum(offs - ons)*mult)
    for tr, (on,of) in enumerate(zip(ons, offs)):
        # extract variables
        neur = frame_by_cluster_matrix[on:of,:] # time x neurons
        this_speed = behave[on:of]
        this_y = y_pos[on:of]
        this_x = x_pos[on:of]

        if interpolation:
            # interpolate over time, double the samples
            current_time = np.arange(len(this_speed))
            new_time = np.arange(0,len(this_speed),.5)
            this_speed = np.interp(new_time, current_time, this_speed)
            this_y = np.interp(new_time, current_time, this_y)
            this_x = np.interp(new_time, current_time, this_x)
            new_neur = np.zeros((len(this_speed),np.shape(neur)[1]))
            for i in np.arange(np.shape(neur)[1]):
                new_neur[:,i] = np.interp(new_time, current_time, neur[:,i])
            neur = new_neur

        if no_stationary:
            # stationary mouse (excluded) # TODO might not work
            moving = this_speed > .5
            neur = neur[moving,:]
            this_speed = this_speed[moving]
            this_x = this_x[moving]
            this_y = this_y[moving]

        # find actual length of time until mouse is in shelter (or 5s if he never makes it)
        in_shelt_y = this_y > session.shelter_location[0][1]
        in_shelt_x = np.logical_and(this_x > session.shelter_location[0][0], this_x < session.shelter_location[1][0])
        in_shelt = np.logical_and(in_shelt_x, in_shelt_y)

        # condition vector
        c = np.zeros((len(this_y)))
        if bar[of] == True: c += 1
        if barflip[of] == True: c += 1

        bin_size = 10
        if compression_var == 'distance_shelter':
            var = compute_dist_shelt(this_x, this_y, c, session)
        elif compression_var == 'y_pos':
            var = this_y
        elif compression_var == 'escape':
            dd = compute_escape_trajectory(this_x, this_y)
            var = (dd/np.amax(dd))
            bin_size = .01 # .01
        elif compression_var == 'speed':
            var = this_speed
            bin_size = 1 # 1

        disc_var = discretize_x_axis(var, bin_size)

        # concatenate trials
        escape_matrix[:,start[-1]:start[-1]+len(disc_var)] = neur.T
        esc_var[start[-1]:start[-1]+len(disc_var)] = disc_var
        in_shelter[start[-1]:start[-1]+len(disc_var)] = in_shelt
        cond[start[-1]:start[-1]+len(disc_var)] = c
        
        start.append(start[-1]+len(disc_var))
        if len(np.where(in_shelt)[0]) == 0:        
            h_start.append(h_start[-1]+len(disc_var)) # never reaches shelter
        else:
            h_start.append(h_start[-1]+len(disc_var[in_shelt == 0])) # only keep homing until mouse reaches shelter
    
    cond = cond[in_shelter == 0]
    esc_var = esc_var[in_shelter == 0]
    escape_matrix = escape_matrix[:,in_shelter == 0]
    
    escape_matrix = zscore(escape_matrix, axis = 1)
    return esc_var, escape_matrix, cond, h_start[:-1]

In [338]:
def creat_tuning_curve(esc_var, escape_matrix, nbins):
    """This function creates a matrix of neurons x bins, where each line is the tuning of that neuron for the variable
    input: escape_matrix is neurons x time
    esc_var is in time"""
    tuning_matrix = np.empty((np.shape(escape_matrix)[0],nbins))
    for i, n in enumerate(escape_matrix):
        tuning_matrix[i,:] = firing_by_bin(esc_var.astype(int), n, nbins)
    return tuning_matrix

def firing_by_bin(var, neural_activity, nbins):
    angles_firing = np.zeros(nbins)
    unique_groups, group_counts = np.unique(var, return_counts=True)
    # mean firing
    group_sums = np.bincount(var, weights=neural_activity)
    angles_firing[unique_groups] = group_sums[unique_groups] / group_counts
    return angles_firing

def smoothed_firing_by_bin(var, neural_activity, nbins):
    # alternative for interpolation
    # neural_activity = escape_matrix[0,:]
    # nbins = int(np.amax(esc_var+1))
    bin_occupancy = np.zeros(nbins)
    bin_sum_activity = np.zeros(nbins)
    unique_groups, group_counts = np.unique(var, return_counts=True)
    group_sums = np.bincount(var, weights=neural_activity)
    angles_firing[unique_groups] = group_sums[unique_groups] / group_counts
    return angles_firing

In [352]:
def create_xval_tuning_curve(esc_var, escape_matrix, start, bins, epoch_method = 'trial', xval_method = 'cosinesim', n_epochs = 3, normalize_tuning_curve = False):
    """Function that computes the tuning of each neuron for a given variable, cross vlaidates it and checks if it's a reliable cell
    several methods can be implemented for xval: separate by trial, even and odd time point, time
    NB: you can call this by condition or for all time!"""
    # divide into epochs
    epochs = np.zeros_like(esc_var)
    if epoch_method == 'time':
        transitions = [int(len(esc_var)/4),int(len(esc_var)/2),int((len(esc_var)/4)*3)]
        for i in transitions:
            epochs[i:] += 1
    if epoch_method == 'alt_time':
        for i in np.arange(1,n_epochs):
            epochs[np.arange(i,len(epochs),n_epochs)] += i
    if epoch_method == 'trial':
        for st in start[1:-1]:
            epochs[st:] += 1
            epochs = np.mod(epochs, n_epochs) # these epochs are not equal in size!!

    # compute tuning curve for each epoch
    result = np.zeros((np.shape(escape_matrix)[0],len(np.unique(epochs))))
    for i in np.unique(epochs):
        test_var = esc_var[epochs == i]
        train_var = esc_var[epochs != i]
        test_mat = escape_matrix[:,epochs == i]
        train_mat = escape_matrix[:,epochs != i]
        test_tuning = creat_tuning_curve(test_var, test_mat, bins)
        train_tuning = creat_tuning_curve(train_var, train_mat, bins)

        # compare tuning curves
        for it in np.arange(len(result)):
            curve1 = test_tuning[it,:]
            curve2 = train_tuning[it,:]
            if normalize_tuning_curve:
                curve1 = test_tuning[it,:]/np.amax(test_tuning[it,:])
                curve2 = train_tuning[it,:]/np.amax(train_tuning[it,:])
                curve1[np.logical_or(test_tuning[it,:] == 0, np.isinf(curve1))] = 0
                curve2[np.logical_or(train_tuning[it,:] == 0, np.isinf(curve2))] = 0
            if xval_method == 'corr':
                result[it,int(i)], _ = pearsonr(curve1, curve2)
            # if xval_method == 'mse': # don't like it, values not spread well
            #     result[it,int(i)] = np.mean((curve1 - curve2)** 2)
            if xval_method == 'cosinesim':
                result[it,int(i)] = cosine_similarity(curve1.reshape(1, -1),curve2.reshape(1, -1))[0,0]
            if xval_method == 'euclid':
                result[it,int(i)] = np.linalg.norm(curve1 - curve2) / np.linalg.norm(curve1 + curve2)

    # average results across epochs
    avg_result = np.mean(result, axis = 1)
    if np.logical_or(xval_method == 'corr', xval_method == 'cosinesim'):
        xval_pass = avg_result > .7
    if xval_method == 'euclid':
        xval_pass = avg_result < .45

    full_tuning = creat_tuning_curve(esc_var, escape_matrix, bins)

    return full_tuning, xval_pass